# SIH26127 — ANPR City-Wide Trajectory Tracking (Colab T4 Demo)

**Runtime target: Google Colab with GPU.** Do not run vision cells on a CPU-only laptop.

## Honest capability statement (read to judges verbatim)
- PROVEN: YOLOv8n vehicle detection on real footage; plate-model loading; EasyOCR on text/overlays; DeepSORT track IDs; track-keyed + per-position voting unit-tested; fusion engine with logged accept/reject reasons; SQLite on Drive.
- NOT YET proven: reliable real number-plate transcription on distant footage (prior: 0 confident reads @ conf≥0.50 across 103 vehicles — 12–20px crop-height limit). Below-gate reads are dropped, never stored.
- Cross-camera fusion runs on **(a) real sightings if produced, else (b) clearly-labelled SYNTHETIC fixtures**. Synthetic sections say `SYNTHETIC DEMO DATA`.
- OCR engine actually used is printed in Cell 9 (EasyOCR primary; PaddleOCR only if it smoke-tests cleanly).

In [ ]:
# CELL 2 — Drive mount (DB, models, crops, HTML all live on Drive)
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/SIH26127_ANPR'
os.makedirs(PROJECT_DIR, exist_ok=True)
DB_PATH = os.path.join(PROJECT_DIR, 'anpr_demo.db')
DEBUG_DIR = os.path.join(PROJECT_DIR, 'debug_crops')
os.makedirs(DEBUG_DIR, exist_ok=True)
print('PROJECT_DIR:', PROJECT_DIR)
print('DB_PATH:', DB_PATH)
print('DEBUG_DIR:', DEBUG_DIR)

In [ ]:
# CELL 3 — GPU verification. If this FAILS: Runtime > Change runtime type > T4 GPU, then re-run.
!nvidia-smi
import torch
print('torch:', torch.__version__, '| cuda_available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'FAILED: no GPU — switch Colab runtime to GPU and re-run'
print('device:', torch.cuda.get_device_name(0))
print('CELL 3 SUCCESS')

In [ ]:
# CELL 4 — Minimal installs (run once, then freeze; never reinstall mid-notebook)
!pip install -q ultralytics easyocr deep-sort-realtime rapidfuzz folium huggingface_hub pandas 2>&1 | tail -n 3
print('installs done — verify in Cell 5')

In [ ]:
# CELL 5 — Version + signature verification (never call from memory)
import inspect, traceback
import torch, cv2, numpy as np
print('torch', torch.__version__, '| cv2', cv2.__version__, '| numpy', np.__version__)
for mod in ['ultralytics', 'easyocr', 'deep_sort_realtime', 'rapidfuzz', 'folium', 'huggingface_hub']:
    try:
        m = __import__(mod); print(mod, getattr(m, '__version__', 'ok'))
    except Exception: traceback.print_exc()
from ultralytics import YOLO
print('YOLO.predict sig:', inspect.signature(YOLO.predict))
import easyocr
print('EasyOCR Reader sig:', inspect.signature(easyocr.Reader.__init__))
print('EasyOCR readtext sig:', inspect.signature(easyocr.Reader.readtext))
from deep_sort_realtime.deepsort_tracker import DeepSort
print('DeepSort sig:', inspect.signature(DeepSort.__init__))
print('update_tracks sig:', inspect.signature(DeepSort.update_tracks))
import huggingface_hub
print('huggingface_hub', huggingface_hub.__version__)
print('hf_hub_download sig:', inspect.signature(huggingface_hub.hf_hub_download))
!pip freeze | grep -i -E 'ultralytics|easyocr|paddle|torch|opencv|deep-sort|rapidfuzz|folium' | tee "$PROJECT_DIR/requirements_colab.txt"
print('CELL 5 SUCCESS')

In [ ]:
# CELL 6 — Repo + DB init (uses fixed resolver from this repo state)
import os, sys, json, glob, shutil
REPO = '/content/SIH'
if not os.path.exists(REPO):
    !git clone -q https://github.com/Parth-debug-cse/SIH.git /content/SIH
cands = glob.glob('/content/SIH/**/src/pipeline_runner.py', recursive=True)
print('pipeline_runner:', cands)
PROJ = os.path.dirname(os.path.dirname(cands[0])) if cands else REPO
print('PROJ:', PROJ)
sys.path.insert(0, PROJ)
CAM_SRC = os.path.join(PROJ, 'data', 'calibration', 'cameras.json')
CAM_DST = os.path.join(PROJECT_DIR, 'cameras.json')
if os.path.exists(CAM_SRC): shutil.copy(CAM_SRC, CAM_DST)
print('cameras:', CAM_DST)
from src.db.schema import init_db
init_db(DB_PATH)
print('DB ready:', DB_PATH)
# Default real footage: WhatsApp video, fallback = any mp4 in PROJECT_DIR
WA = os.path.join(PROJECT_DIR, 'WhatsApp Video 2026-09-11 at 1.35.19 PM.mp4')
VIDEO_PATH = os.environ.get('VIDEO_PATH', WA)
if not os.path.exists(VIDEO_PATH):
    v = sorted(glob.glob(PROJECT_DIR + '/*.mp4'))
    VIDEO_PATH = v[0] if v else ''
print('VIDEO_PATH:', VIDEO_PATH or 'NONE — upload footage to PROJECT_DIR')
print('CELL 6 SUCCESS')

In [ ]:
# CELL 7 — Model resolution + loading smoke test (HARD FAIL on any error)
# Fixes the blobs/ TypeError: resolver must return a real .pt file.
import traceback, hashlib
from pathlib import Path
try:
    from src.detection.detector import PlateDetector, resolve_plate_model_from_hub
    pw = resolve_plate_model_from_hub()  # prints resolved/suffix/size/sha256
    assert pw.suffix == '.pt', f"FAILED: not a .pt path: {pw}"
    assert pw.stat().st_size > 0, 'FAILED: empty weights'
    print('stable .pt verified:', pw, pw.stat().st_size, 'bytes')
    DEVICE = 'cuda:0'
    det = PlateDetector(device=DEVICE, plate_model_path=pw)
    print('PlateDetector OK | plate gate:', det.plate_conf_threshold,
          '| aspect:', det.plate_aspect_ratio_bounds)
    import numpy as np
    probe = (np.random.rand(480, 640, 3) * 255).astype(np.uint8)
    res = det._vehicle_model.predict(source=probe, conf=0.25, device=DEVICE, verbose=False)
    r0 = res[0]
    print('Results type:', type(r0).__name__)
    print('has .boxes:', r0.boxes is not None)
    if r0.boxes is not None and len(r0.boxes) > 0:
        b = r0.boxes[0]
        print('xyxy:', b.xyxy[0].cpu().numpy().tolist(), '| conf:', float(b.conf[0].item()), '| cls:', int(b.cls[0].item()))
    else:
        print('probe: no boxes on noise (expected) — API shape verified via attrs:', [a for a in ('xyxy','conf','cls') if hasattr(r0.boxes, a)] if r0.boxes is not None else 'boxes None')
    print('CELL 7 SUCCESS — model loads, CUDA inference runs, Results inspected')
except Exception:
    traceback.print_exc()
    raise AssertionError('CELL 7 FAILED — fix model resolution before continuing')

In [ ]:
# CELL 8 — ONE real-frame detection diagnostic (vehicles -> in-crop plates -> full-frame fallback)
import traceback, cv2
try:
    assert VIDEO_PATH and os.path.exists(VIDEO_PATH), 'FAILED: no video file'
    cap = cv2.VideoCapture(VIDEO_PATH)
    ok, frame = cap.read(); cap.release()
    assert ok, 'FAILED: cannot read frame 0'
    print('frame shape:', frame.shape)
    vehs = det.detect_vehicles(frame)
    print('vehicles detected:', len(vehs))
    for v in vehs[:5]: print('  ', v['bbox'], v['class_name'], round(v['confidence'], 2))
    det.reset_plate_aspect_ratio_counter()
    in_crop, full = 0, []
    for v in vehs:
        d = det.detect_plates_in_crop(frame, v['bbox'])
        in_crop += len(d)
        for p in d: print('  in-crop plate:', p['bbox'], 'conf=', round(p['confidence'], 2))
    if in_crop == 0 and vehs:
        full = det.detect_plates(frame, vehicle_detections=vehs)
        print('full-frame fallback plates:', len(full))
        for p in full[:5]: print('  ff plate:', p['bbox'], 'conf=', round(p['confidence'], 2), 'veh_idx=', p['vehicle_idx'])
    print('aspect-ratio rejects:', det.plate_aspect_ratio_rejects, '| position rejects:', det.plate_position_rejects)
    ann = frame.copy()
    for v in vehs:
        x1, y1, x2, y2 = v['bbox']; cv2.rectangle(ann, (x1, y1), (x2, y2), (0, 255, 0), 2)
    for p in full:
        x1, y1, x2, y2 = p['bbox']; cv2.rectangle(ann, (x1, y1), (x2, y2), (0, 0, 255), 2)
    cv2.imwrite(os.path.join(DEBUG_DIR, 'annotated_frame0.jpg'), ann)
    print('annotated frame saved to', DEBUG_DIR)
    print('CELL 8 SUCCESS')
except Exception:
    traceback.print_exc()
    raise AssertionError('CELL 8 FAILED')

In [ ]:
# CELL 9 — Real-crop OCR diagnostic (3 preprocessing variants, verified EasyOCR API)
import traceback, cv2, numpy as np
OCR_ENGINE = 'none'
try:
    from src.ocr.reader import PlateOCR, looks_like_plate
    ocr = PlateOCR(use_gpu=True)
    print('EasyOCR Reader created with verified kwargs: lang, gpu, verbose')
    r = ocr.reader.readtext(np.zeros((60, 200, 3), np.uint8))
    print('readtext return type on blank:', type(r).__name__, 'len:', len(r))
    OCR_ENGINE = 'easyocr'
    # Build a test crop: best full-frame plate box, else largest vehicle
    boxes = [p['bbox'] for p in full] if full else []
    if boxes:
        x1, y1, x2, y2 = boxes[0]; src, tag = 'plate_box', 'plate'
    else:
        v = max(vehs, key=lambda v: (v['bbox'][3]-v['bbox'][1])); x1, y1, x2, y2 = v['bbox']; src, tag = 'vehicle_fallback', 'veh'
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = max(0,x1), max(0,y1), min(w,x2), min(h,y2)
    crop = frame[y1:y2, x1:x2]
    print(f'crop source={src} raw={crop.shape[1]}x{crop.shape[0]}')
    cv2.imwrite(os.path.join(DEBUG_DIR, 'crop_raw.jpg'), crop)
    v2 = ocr.preprocess_plate(crop)  # CLAHE + INTER_CUBIC 3x + unsharp
    print('preprocessed:', v2.shape)
    cv2.imwrite(os.path.join(DEBUG_DIR, 'crop_clahe_upsharp.jpg'), v2)
    for name, img in [('raw_up', cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)), ('clahe_upsharp', v2)]:
        out = ocr.read_plate(img)
        print(f"{name}: text='{out['text']}' conf={out['confidence']:.3f} raw='{out['raw_text']}'")
    print('OCR_ENGINE =', OCR_ENGINE)
    print('CELL 9 SUCCESS')
except Exception:
    traceback.print_exc()
    raise AssertionError('CELL 9 FAILED')

In [ ]:
# CELL 10 — Short REAL VIDEO pipeline (frame_skip=2) with stage-by-stage report
import traceback, cv2
try:
    from src.pipeline_runner import PipelineRunner
    import src.db.schema as _sch
    from pathlib import Path as _P
    from src.db.schema import init_db
    init_db(DB_PATH)
    _old, _sch.DB_PATH = _sch.DB_PATH, _P(DB_PATH)
    FRAME_SKIP = 2
    N_FRAMES = int(os.environ.get('N_FRAMES', '60'))
    runner = PipelineRunner(camera_id='cam_wa', video_path=VIDEO_PATH,
        camera_config={'gps_lat': 12.9758, 'gps_lon': 77.6082}, device='cuda:0',
        debug_crops_dir=DEBUG_DIR)
    cap = cv2.VideoCapture(VIDEO_PATH)
    fno, proc, tv, tp = 0, 0, 0, 0
    try:
        while True:
            ok, fr = cap.read()
            if not ok or fno >= N_FRAMES: break
            if fno % FRAME_SKIP == 0:
                rr = runner.process_frame(fr, fno)
                proc += 1; tv += rr['vehicles_detected']; tp += rr['plates_read']
                if proc <= 5:
                    print(f"frame {fno}: vehicles={rr['vehicles_detected']} plates_entries={len(rr['plates'])} plates_read={rr['plates_read']} tracks={len(rr['tracked'])}")
            fno += 1
    finally:
        cap.release(); _sch.DB_PATH = _old
    d = runner._diag
    print(f"scanned={fno} processed={proc} vehicles={tv} plates_read={tp}")
    print('vehicles_with_plate_box:', d['vehicles_with_plate_box'], '| no_plate_box:', d['no_plate_box_found'])
    print('fallback_hits:', d['plate_fullframe_fallback_used'], '| ff_boxes:', d['plate_fullframe_boxes'])
    print('tiny_rejected:', d['plate_crop_tiny_skipped'], '| below_gate:', d['plate_found_below_gate'])
    print('gate_cleared:', d['plate_gate_cleared'], '| format_rejected:', d['plate_confidence_cleared_format_rejected'])
    print('crop heights:', d['plate_crop_heights'][:12])
    print('CELL 10 SUCCESS')
except Exception:
    traceback.print_exc()
    raise AssertionError('CELL 10 FAILED')

In [ ]:
# CELL 10b — Candidate-ledger diagnosis (thresholds UNCHANGED, diagnostics only)
import traceback, os
try:
    import pandas as pd
    rows = list(getattr(runner, '_ledger', [])) or None
    if not rows:
        import glob as _g
        csvs = _g.glob(os.path.join(DEBUG_DIR, 'candidate_ledger_*.csv'))
        assert csvs, 'FAILED: no in-memory ledger and no CSV — run Cell 10 first'
        df = pd.read_csv(sorted(csvs)[-1])
        print('loaded ledger CSV:', sorted(csvs)[-1])
    else:
        df = pd.DataFrame(rows)
    print('candidates:', len(df))
    print('\n-- plate-detector confidence distribution --')
    print(df['det_conf'].describe()[['count', 'mean', 'min', 'max']])
    print('det threshold:', df['det_threshold'].iloc[0])
    print('\n-- OCR accept-confidence distribution (non-empty reads) --')
    occ = df[df['accept_text'].astype(bool)]['accept_conf']
    print(occ.describe()[['count', 'mean', 'min', 'max']] if len(occ) else 'no non-empty OCR reads')
    print('ocr threshold:', df['ocr_threshold'].iloc[0])
    print('\n-- gate survival --')
    for k in ['candidate_boxes', 'ocr_attempted', 'ocr_empty', 'ocr_low_conf', 'regex_rejected', 'accepted']:
        print(f"  {k} = {runner._diag.get(k, 'n/a')}")
    print('\n-- top rejection reasons --')
    print(df['final_reason'].value_counts())
    print('\n-- raw vs preprocessed OCR agreement --')
    agree = (df['ocr_raw_text'] == df['ocr_pre_text']).mean() if len(df) else float('nan')
    print(f'raw==pre text agreement: {agree:.2f}')
    print(df[['frame', 'det_conf', 'crop_w', 'crop_h', 'ocr_raw_text', 'ocr_raw_conf', 'ocr_pre_text', 'ocr_pre_conf', 'accept_text', 'accept_conf', 'corrected', 'regex_pass', 'final_reason']].head(20).to_string())
    print('CELL 10b SUCCESS')
except Exception:
    traceback.print_exc()
    raise AssertionError('CELL 10b FAILED')


In [ ]:
# CELL 10c — Direct-recognition experiment (whole-crop recognize, thresholds UNCHANGED)
# Compares readtext() baseline vs direct recognize() over variants A–F on the SAME crops.
import traceback, os, inspect
try:
    import pandas as pd
    print('recognize sig:', inspect.signature(ocr.reader.recognize))
    trials = list(getattr(runner, '_variant_trials', [])) or None
    if not trials:
        import glob as _g
        csvs = _g.glob(os.path.join(DEBUG_DIR, 'top5', 'top5_direct.csv'))
        assert csvs, 'FAILED: no trials in memory and no top5 CSV — run Cell 10 first'
        df = pd.read_csv(sorted(csvs)[-1])
        print('loaded:', sorted(csvs)[-1])
    else:
        df = pd.DataFrame(trials)
    ok = df[df['status'] == 'ok']
    print(f"trials={len(df)} ok={len(ok)} variants={sorted(df['variant'].unique())}")
    print('\n-- mean conf by variant (ok trials) --')
    print(ok.groupby('variant')['conf'].agg(['count', 'mean', 'max']).round(3).sort_values('mean', ascending=False) if len(ok) else 'none')
    print('\n-- top 20 observations by OCR confidence --')
    cols = ['frame', 'det_conf', 'variant', 'text', 'conf', 'normalized', 'regex_pass', 'status']
    print(ok.sort_values('conf', ascending=False)[cols].head(20).to_string() if len(ok) else 'none')
    print('\n-- readtext() baseline accept-confs for comparison --')
    led = pd.DataFrame(list(getattr(runner, '_ledger', [])))
    if len(led):
        print(led[led['accept_text'].astype(bool)].sort_values('accept_conf', ascending=False)[['frame', 'det_conf', 'accept_text', 'accept_conf', 'best_variant', 'best_variant_text', 'best_variant_conf', 'final_reason']].head(10).to_string())
    print('\ntop-5 crops saved under:', os.path.join(DEBUG_DIR, 'top5'))
    print('CELL 10c SUCCESS')
except Exception:
    traceback.print_exc()
    raise AssertionError('CELL 10c FAILED')


In [ ]:
# CELL 11 — Multi-frame fusion unit test (per-position weighted vote, deterministic)
from src.ocr.reader import fuse_track_observations
obs = [('KA01AB1234', 0.91), ('KA01AB1234', 0.84), ('KA01AB1284', 0.51), ('KA01AB1234', 0.76), ('KA01AB123', 0.60)]
res = fuse_track_observations(obs)
print(res)
assert res['canonical_plate'] == 'KA01AB1234' and res['method'] == 'per_position_weighted', 'CELL 11 FAILED'
print('CELL 11 SUCCESS')

In [ ]:
# CELL 12 — Cross-camera fusion (real sightings if >=2, else LABELLED synthetic)
import traceback, time
from src.fusion.engine import load_sightings, run_fusion_once, store_trajectories
from src.fusion.reid import CrossCameraFusion
import src.db.schema as _sch
from pathlib import Path
_sch.DB_PATH, _o = Path(DB_PATH), _sch.DB_PATH
try:
    real = load_sightings(DB_PATH)
    print('real sightings:', len(real))
    if len(real) >= 2:
        print('REAL fusion path')
        print(run_fusion_once(db_path=DB_PATH, cameras_config_path=CAM_DST))
    else:
        print('== SYNTHETIC DEMO DATA (not real cross-camera footage) ==')
        t0 = time.time()
        synth = [
            {'plate': 'KA01AB1234', 'confidence': 0.88, 'camera_id': 'cam_1', 'gps_lat': 12.9758, 'gps_lon': 77.6082, 'timestamp': t0, 'vehicle_class': 'car', 'track_id': 1, 'direction': '45deg'},
            {'plate': 'KA01AB1234', 'confidence': 0.81, 'camera_id': 'cam_2', 'gps_lat': 12.9768, 'gps_lon': 77.6050, 'timestamp': t0 + 250, 'vehicle_class': 'car', 'track_id': 3, 'direction': '120deg'},
            {'plate': 'KA05MN9999', 'confidence': 0.90, 'camera_id': 'cam_2', 'gps_lat': 12.9768, 'gps_lon': 77.6050, 'timestamp': t0 + 5, 'vehicle_class': 'bus', 'track_id': 9, 'direction': '120deg'},
        ]
        fz = CrossCameraFusion(cameras_config_path=CAM_DST)
        for i in range(len(synth)):
            for j in range(i + 1, len(synth)):
                a = fz.associate_sightings(synth[i], synth[j]); dd = a.to_dict()
                print(f"{dd['plate_a']}@{dd['camera_a']} -> {dd['plate_b']}@{dd['camera_b']} sim={dd['similarity']:.1f} dt={a.elapsed_time:.0f}s dist={a.distance_km:.3f}km speed={a.speed_kmh:.1f} accepted={dd['accepted']} reason={dd['hard_reject_reason']}")
        tj = fz.fuse_trajectories(synth)
        store_trajectories(tj, db_path=DB_PATH, source='colab_synthetic_demo')
        print('stored', len(tj), 'synthetic trajectories')
    print('CELL 12 SUCCESS')
except Exception:
    traceback.print_exc()
    raise AssertionError('CELL 12 FAILED')
finally:
    _sch.DB_PATH = _o

In [ ]:
# CELL 13 — Trajectory query
import sqlite3
QUERY_PLATE = os.environ.get('QUERY_PLATE', 'KA01AB1234')
con = sqlite3.connect(DB_PATH); con.row_factory = sqlite3.Row
try:
    rows = list(con.execute('SELECT id, trajectory_code, plate, route, num_sightings FROM trajectories WHERE plate=?', (QUERY_PLATE,)))
    print(f'trajectories for {QUERY_PLATE}:', len(rows))
    for t in rows:
        print(dict(t))
        for s in con.execute('SELECT plate, camera_id, gps_lat, gps_lon, timestamp, seq FROM trajectory_sightings WHERE trajectory_id=? ORDER BY seq', (t['id'],)):
            print('  ', dict(s))
finally: con.close()
print('CELL 13 SUCCESS')

In [ ]:
# CELL 14 — Map (folium -> Drive HTML)
import traceback, sqlite3
try:
    import folium
    con = sqlite3.connect(DB_PATH); con.row_factory = sqlite3.Row
    tid = con.execute('SELECT id FROM trajectories WHERE plate=? ORDER BY id DESC LIMIT 1', (QUERY_PLATE,)).fetchone()
    assert tid is not None, 'no trajectory — run Cell 12 first'
    pts = list(con.execute('SELECT camera_id, gps_lat, gps_lon, timestamp FROM trajectory_sightings WHERE trajectory_id=? ORDER BY seq', (tid['id'],))); con.close()
    m = folium.Map(location=[pts[0]['gps_lat'], pts[0]['gps_lon']], zoom_start=14)
    for i, p in enumerate(pts):
        folium.Marker([p['gps_lat'], p['gps_lon']], popup=f"{i+1}. {p['camera_id']} @ {p['timestamp']:.0f}").add_to(m)
    folium.PolyLine([[p['gps_lat'], p['gps_lon']] for p in pts], color='blue').add_to(m)
    out = PROJECT_DIR + f'/trajectory_{QUERY_PLATE}.html'
    m.save(out); print('saved:', out)
    m
except Exception:
    traceback.print_exc()
    raise AssertionError('CELL 14 FAILED')

In [ ]:
# CELL 15 — Analytics (pandas over Drive DB)
import sqlite3, pandas as pd
con = sqlite3.connect(DB_PATH)
df = pd.read_sql_query('SELECT camera_id, plate, timestamp FROM sightings', con); con.close()
print('sightings per camera:'); print(df.groupby('camera_id').size() if not df.empty else 'no sightings (honest zero)')
con = sqlite3.connect(DB_PATH)
od = pd.read_sql_query('SELECT plate, route FROM trajectories', con); con.close()
print('OD routes:'); print(od['route'].value_counts() if not od.empty else od)
print('CELL 15 SUCCESS')

In [ ]:
# CELL 16 — Blacklist alert demo (labelled DEMO)
import json, time
from src.alerts.engine import AlertEngine
bl = os.path.join(PROJECT_DIR, 'blacklist_demo.json')
json.dump(['KA01AB1234'], open(bl, 'w'))
eng = AlertEngine(blacklist_path=bl)
print('DEMO blacklist alerts:', eng.check_plate(plate='KA01AB1234', camera_id='cam_2', gps_lat=12.9768, gps_lon=77.6050, timestamp=time.time()))
print('CELL 16 SUCCESS')

In [ ]:
# CELL 17 — Final summary + acceptance checklist
print('DB:', DB_PATH, '| video:', VIDEO_PATH, '| OCR:', OCR_ENGINE)
!ls -la "$PROJECT_DIR" | head -n 20
print('debug crops:', len(__import__('glob').glob(DEBUG_DIR + '/*.jpg')))
print("""ACCEPTANCE: [.pt resolve] [YOLO CUDA] [Results inspected] [1-frame vehicles+plates] [crops saved] [OCR engine printed+shaped] [video stages] [per-position test] [fusion reasons] [DB on Drive] [map] [analytics] [alerts] — verify each cell printed SUCCESS above.""")